# 06 Experiments Over Baseline 3
In this notebook, I will experiment with a decoder-only model for prediction, specifically the LLaMA-3.1 model. Then, I will perform a test of the ensemble of the FinBERT and LLaMA models. The reasoning is as follows: We've seen previously (and explained) why an encoder-only model didn't work. However, a decoder-only model (to understand the whole transcript and generate a response for me) may be subject to hallucinations and simply generate information based on my prompt, meaning it doesn't necessarily generate based on what it understands about the underlying tones of the transcripts. Therefore, I will try an ensemble of these models and see how accurate their agreed predictions are.

# Imports

In [ ]:
!pip install gensim nltk evaluate transformers datasets

!pip install -q -U bitsandbytes
# !pip install -q -U bitsandbytes flash_attn
!pip install flash_attn==2.7.4.post1 --no-build-isolation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 126.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 110.3 MB/s eta 0:00:00
  Using cached flash_attn-2.7.4.post1.tar.gz (6.0 MB)
  Preparing metadata (setup.py) ... done
  Created wheel for flash_

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date
from tqdm import tqdm
import os
import warnings
from pprint import pprint

import sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

In [ ]:
from gensim.models import Word2Vec
import nltk
from nltk.util import ngrams
from nltk.tokenize import word_tokenize
from collections import defaultdict, Counter
import evaluate

import tensorflow as tf
from keras.models import Model
from keras.layers import Input, Embedding, Lambda, Dense
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets import Dataset, load_dataset

from transformers import BertTokenizer, BertModel, BertPreTrainedModel, BertForSequenceClassification
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, BertConfig
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from transformers import pipeline, BitsAndBytesConfig, AutoModelForCausalLM

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## Loading in full dataset

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Changing directory to get the data
# data_path = '/content/drive/MyDrive/MIDS DATASCI 266/MIDS DATASCI 266 Final Project'
data_path = '/content/drive/MyDrive/Work Stuff/Berkeley MIDS Stuff/Berkeley MIDS 266 Stuff/w266_project'
os.chdir(data_path)
os.getcwd()

'/content/drive/MyDrive/Work Stuff/Berkeley MIDS Stuff/Berkeley MIDS 266 Stuff/w266_project'

In [4]:
# Retrieving data
full_data_path = 'SNP500_Transcripts_Price_2015_to_2024.csv'
full_data = pd.read_csv(full_data_path, sep='|', index_col=0)

In [5]:
# Limiting the data to only the text and the chosen target variable
model_df = full_data[['Text', 'Close_5_dir']].copy()
model_df['Close_5_dir'] = model_df['Close_5_dir'].astype(np.int64)

model_df

,Text,Close_5_dir
0,"Good afternoon. My name is Karen, and I'll be ...",0
1,"Ladies and gentlemen, thank you for standing b...",1
2,"Good day, ladies and gentlemen, and welcome to...",0
3,"Good morning, ladies and gentlemen, and welcom...",1
4,"Good morning, ladies and gentlemen, and welcom...",1
...,...,...
17228,"Good day, and thank you for standing by. Welco...",0
17229,Welcome to Lennar's Fourth Quarter Earnings Co...,0
17230,"Good afternoon, everyone. Welcome to NIKE, Inc...",0
17231,"Good day, and welcome to the FedEx Fiscal Year...",0


# Llama Model Experiments
Instead of only using encoder models, I'll also experiment with some large decoder models like Llama and see if I can obtain better results. Specifically, I'll be experimenting with the Meta Llama 3.1 model [here](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct).

In [ ]:
# Quantization to shrink the memory footprint of the LLM, allowing us to load it on a smaller GPU
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

## Prompt engineering with one transcript
I'll experiment with one transcript to get the prompt correct.

In [ ]:
# Getting the model id
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Getting a pipeline for text generation based on what Llama thinks about the overall sentiment about the earnings call transcript
pipeline = pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16, "quantization_config": quantization_config},
    device_map="auto",
)

# Writing the message to the model
text_0 = model_df['Text'][0]
content_message = f"Based on the following earnings call transcript, what is the likely one-week market direction of the stock if entered at the market open on the next business day?\n\nTranscript:\n{text_0}\n\nPlease respond in this format:\n{{\"prediction\": \"confident up\"}}\nValid outputs are: confident up, confident down, unconfident up, unconfident down."
messages = [
    {"role": "system", "content": "You are a financial analyst assistant trained to interpret earnings call transcripts and predict near-term market sentiment."},
    {"role": "user", "content": content_message}
]

# Getting the outputs through a pipeline
outputs = pipeline(messages, max_new_tokens=128)

pprint(outputs[0]["generated_text"][-1], compact=True)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


{'content': '{"prediction": "confident up"}\n'
            '\n'
            'The earnings call transcript indicates a positive outlook for the '
            'company, with strong revenue and bit sales volume growth in the '
            'first quarter. The CEO, Mark Durcan, mentioned that the company '
            'is benefiting from favorable market conditions and solid '
            'execution from the team. He also stated that the company is '
            'expecting continued favorable market conditions for 2015, led by '
            'constrained supply in DRAM and solid demand for both the DRAM and '
            'NAND.\n'
            '\n'
            'The guidance for the second quarter also suggests a positive '
            'outlook, with revenue expected to be in the range of $4.1 billion '
            'to $4.3 billion, which reflects the',
 'role': 'assistant'}


Clearly, that did not generate the signal for me. Thus, I'll spend some time trying different prompts and see what results they give me.

In [ ]:
# Trying again but with a different prompt
text_1 = model_df['Text'][1]

# Getting the system and user messages
system_message = "You are a financial analyst assistant trained to interpret earnings call transcripts and predict near-term market sentiment using underlying tone and word choice."
user_message = f"""Based on the following earnings call transcript, what is the likely one-week market direction of the stock if entered at the market open on the next business day? Keep in mind that a seemingly positive sentiment (which most transcripts try to express) might be hiding more negative undertones. It also might not be, but make sure to pay attention to those.
You are only allowed to respond with 2 words, which can be one of the following: confident up, confident down, unconfident up, unconfident down. When selecting your 2-word response, keep in mind the following:
- confident up means that the underlying tone is still positive, and the speaker is showing little signs of hesitation or weakness, meaning you should predict confidently that the market direction will go up
- unconfident up means that most of the transcript is confident, but perhaps with some minor points that could be hinting that not everything is as perfect as it seems
- unconfident down means that the earnings call may project confidence, but the tone implies otherwise, and there may be a bit more hesitation
- confident down covers everything else, where even if the transcript is positive overall, if there's more hesitation, if the underlying tone suggests the speakers are hiding something, if key points are diminished, or if their numbers missed the mark, the stock is likely to go in a down direction over the next week

Transcript:
{text_1}"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message}
]

# Getting the outputs through a pipeline
outputs_1 = pipeline(messages, max_new_tokens=5)

pprint(outputs_1[0]["generated_text"][-1], compact=True)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


{'content': 'confident up', 'role': 'assistant'}


Finally, I've acehived a prompt that seems to be better, and the predictions are not only in the format that I want, but they're also diverse (as they should be) and not just generating "confident up" all the time.

Therefore, I'll continue with the above prompt for now, turn the whole thing into a function, and get predictions for a training set of transcripts. If necessary, I'll also try updating the prompts and see if that improves my training on my full dataset.

## Prompt engineering with small train dataset
As mentioned, I'll run through a small sample of train data, updating my prompt as needed make sure I get the most optimal prompts possible.

In [ ]:
# Defining a function to run through all the transcripts in the training dataset so I can update the prompts
def train_prompt_text(text_df, pipeline, system_message, user_message):

    # Initializing a dictionary
    results_dict = {}

    # Looping through the transcripts
    for idx, transcript in tqdm(zip(text_df.index, text_df['Text'])):

        # Getting the full message:
        full_user_message = user_message + transcript
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": full_user_message}]

        # Initializing a list of generations
        results_list = []

        # Getting three generations
        for _ in range(3):
            outputs = pipeline(messages, max_new_tokens=5)
            gen_results = outputs[0]["generated_text"][-1]['content']
            results_list.append(gen_results)

        results_dict[idx] = results_list

    return results_dict


In [ ]:
# Splitting the data using train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    model_df[['Text']],
    model_df['Close_5_dir'],
    test_size=0.2,
    random_state=42,
    stratify=model_df['Close_5_dir']
)

In [ ]:
# Testing the Llama-3.1 confident/unconfident up and down predictions based on prompt above
# Starting with just 10 transcripts
X_train_small = X_train[:10]

# Getting the system and user messages
system_message = "You are a financial analyst assistant trained to interpret earnings call transcripts and predict near-term market sentiment using underlying tone and word choice."
user_message = """Based on the following earnings call transcript, what is the likely one-week market direction of the stock if entered at the market open on the next business day? Keep in mind that a seemingly positive sentiment (which most transcripts try to express) might be hiding more negative undertones. It also might not be, but make sure to pay attention to those.
You are only allowed to respond with 2 words, which can be one of the following: confident up, confident down, unconfident up, unconfident down. When selecting your 2-word response, keep in mind the following:
- confident up means that the underlying tone is still positive, and the speaker is showing little signs of hesitation or weakness, meaning you should predict confidently that the market direction will go up
- unconfident up means that most of the transcript is confident, but perhaps with some minor points that could be hinting that not everything is as perfect as it seems
- unconfident down means that the earnings call may project confidence, but the tone implies otherwise, and there may be a bit more hesitation
- confident down covers everything else, where even if the transcript is positive overall, if there's more hesitation, if the underlying tone suggests the speakers are hiding something, if key points are diminished, or if their numbers missed the mark, the stock is likely to go in a down direction over the next week

Transcript:
"""

results_dict = train_prompt_text(X_train_small, pipeline, system_message, user_message)

results_dict

0it [00:00, ?it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
1it [00:07,  7.72s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
2it [00:11,  5.22s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
3it [00:17,  5.49s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
4it [00:21,  4.99s/it]Setting `pad_token

{4977: ['confident up', 'unconfident down', 'confident up'],
 9509: ['unconfident up', 'confident up', 'confident up'],
 6714: ['unconfident up', 'confident up', 'confident up'],
 589: ['confident up', 'unconfident up', 'confident down'],
 15539: ['confident up', 'confident up', 'unconfident up'],
 6126: ['confident up', 'confident up', 'confident up'],
 757: ['confident up', 'unconfident up', 'confident up'],
 3916: ['unconfident up', 'confident up', 'unconfident up'],
 3646: ['unconfident down', 'confident up', 'unconfident down'],
 2957: ['unconfident up', 'confident up', 'unconfident up']}

While this was just a test for 10 results, we can get a quick sense of how accurate these results were.

In [ ]:
y_train[:10]

,Close_5_dir
4977,0
9509,1
6714,1
589,1
15539,0
6126,1
757,0
3916,1
3646,0
2957,1


Incredibly, we're actually able to see some semblance of the model coming together. Assuming we take an average of the responses above, we get roughly ~6-7 right, depending on how we want to measure these responses.

This was after hours of prompt engineering to get these diverse responses, and I noticed in my experiments that it's difficult to get the model to predict down. When I tried to force more "down" responses, the model lost all nuance and predicted "down" much more frequently. Therefore, I'll stick with my current prompt and the results it generates, and I'll need to figure out the weights of each of the 4 responses.

In fact, what we can do next after getting these predictions into a csv is to train another model to take those predictions and find the best weights for our 4 confidence levels.

## Saving predictions from prompt engineering into csv
The goal is to get all of our predictions from our dataset into one csv. Normally, I would split the train and test data, but since I've already settled on my prompt, the train and test predictions will be done the same way anyway.

What needs to be trained further is my class weighting, but that can be done after I get my full dataset.

In [ ]:
# Getting the model id
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Getting a pipeline for text generation based on what Llama thinks about the overall sentiment about the earnings call transcript
pipeline = pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16, "quantization_config": quantization_config},
    device_map="auto",
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
# Initializing the path to the direction results
dir_results_path = './llama_direction_results.csv'

# Getting the transcript id in the dataframe
llama_df = model_df.reset_index(names='Transcript_Num')

In [ ]:
# Defining a function to run through all the transcripts and save the direction predictions from Llama-3.1
def save_llama_predictions(text_df, pipeline, system_message, user_message):

    # Reading from the saved csv
    saved_dir_results = pd.read_csv(dir_results_path, index_col='Unnamed: 0')

    # Initializing a df
    results_df = pd.DataFrame(columns=['Transcript_Num', 'pred_1', 'pred_2', 'pred_3'])

    # Looping through the transcripts
    for transcript_num, transcript in tqdm(zip(text_df['Transcript_Num'], text_df['Text']), total=len(text_df)):

        # Skipping if the result already exists
        if transcript_num in saved_dir_results['Transcript_Num']:
            continue

        # Getting the full message:
        full_user_message = user_message + transcript
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": full_user_message}]

        # Initializing a list of generations
        results_list = [transcript_num]

        # Getting three generations
        for _ in range(3):
            outputs = pipeline(messages, max_new_tokens=5)
            gen_results = outputs[0]["generated_text"][-1]['content']
            results_list.append(gen_results)

        # Adding the results to the df
        saved_dir_results.loc[len(saved_dir_results)] = results_list
        # saved_dir_results = saved_dir_results.reset_index(drop=True)
        saved_dir_results.to_csv(dir_results_path)


In [ ]:
# Getting the system and user messages the same as before (after prompt engineering)
system_message = "You are a financial analyst assistant trained to interpret earnings call transcripts and predict near-term market sentiment using underlying tone and word choice."
user_message = """Based on the following earnings call transcript, what is the likely one-week market direction of the stock if entered at the market open on the next business day? Keep in mind that a seemingly positive sentiment (which most transcripts try to express) might be hiding more negative undertones. It also might not be, but make sure to pay attention to those.
You are only allowed to respond with 2 words, which can be one of the following: confident up, confident down, unconfident up, unconfident down. When selecting your 2-word response, keep in mind the following:
- confident up means that the underlying tone is still positive, and the speaker is showing little signs of hesitation or weakness, meaning you should predict confidently that the market direction will go up
- unconfident up means that most of the transcript is confident, but perhaps with some minor points that could be hinting that not everything is as perfect as it seems
- unconfident down means that the earnings call may project confidence, but the tone implies otherwise, and there may be a bit more hesitation
- confident down covers everything else, where even if the transcript is positive overall, if there's more hesitation, if the underlying tone suggests the speakers are hiding something, if key points are diminished, or if their numbers missed the mark, the stock is likely to go in a down direction over the next week

Transcript:
"""

# Running through all the direction predictions from Llama-3.1 and saving them onto the csv
save_llama_predictions(llama_df, pipeline, system_message, user_message)

Streaming output truncated to the last 5000 lines.
 90%|█████████ | 15567/17233 [6:21:53<2:53:57,  6.26s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
 90%|█████████ | 15568/17233 [6:22:00<2:56:22,  6.36s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
 90%|█████████ | 15569/17233 [6:22:08<3:14:29,  7.01s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
 90%|█████████ | 15570/17233 [6:22:14<3:02:50,  6.60s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end gen

In [ ]:
# Disconnecting the colab runtime once finished
# from google.colab import runtime
# runtime.unassign()

In [ ]:
# Getting the direction results back to check it
dir_results_path = './llama_direction_results.csv'
dir_results_df = pd.read_csv(dir_results_path, index_col='Unnamed: 0')

dir_results_df

,Transcript_Num,pred_1,pred_2,pred_3
0,0,unconfident down,unconfident up,unconfident up
1,1,confident up,unconfident up,confident up
2,2,confident up,confident up,unconfident down
3,3,confident up,confident up,confident up
4,4,unconfident down,confident up,confident up
...,...,...,...,...
17228,17228,unconfident down,confident up,confident up
17229,17229,unconfident down,unconfident down,unconfident down
17230,17230,unconfident up,unconfident down,confident down
17231,17231,confident up,unconfident up,confident down


# Analyzing Llama-3.1 confidence predictions
Now that we've gotten our three predictions for each of the transcripts, we can use those scores to get an up or down direction prediction. We'll first do clean our data and do some quick EDA on the predictions, and then train a model for our task.

## Cleaning the data

In [ ]:
# Obtaining the direction results
dir_results_path = './llama_direction_results.csv'
dir_results_df = pd.read_csv(dir_results_path, index_col='Unnamed: 0')

dir_results_df

,Transcript_Num,pred_1,pred_2,pred_3
0,0,unconfident down,unconfident up,unconfident up
1,1,confident up,unconfident up,confident up
2,2,confident up,confident up,unconfident down
3,3,confident up,confident up,confident up
4,4,unconfident down,confident up,confident up
...,...,...,...,...
17228,17228,unconfident down,confident up,confident up
17229,17229,unconfident down,unconfident down,unconfident down
17230,17230,unconfident up,unconfident down,confident down
17231,17231,confident up,unconfident up,confident down


While most of our predictions are among {confident up, unconfident up, confident down, unconfident down}, there still might be some messy predictions. Therefore, let's clean those first before anything else.

In [ ]:
# Getting a cleaned results df
cleaned_results = dir_results_df.copy()

# Stacking into column
cleaned_results[['pred_1', 'pred_2', 'pred_3']].stack().value_counts()

,count
confident up,19747
unconfident up,17065
unconfident down,11688
confident down,3045
unconfident up \n\n,55
unconfident up.,36
unconfident down \n\n,23
unconfident up\n\n,9
unconfident down.,8
unconfident down\n\n,6


Let's make everything lowercase, then truncate to just two words, then remove any \n, periods, or other add-ons.

In [ ]:
# Making the predictions lowercase
cleaned_results['pred_1'] = cleaned_results['pred_1'].str.lower()
cleaned_results['pred_2'] = cleaned_results['pred_2'].str.lower()
cleaned_results['pred_3'] = cleaned_results['pred_3'].str.lower()

# Truncate to just two words
cleaned_results['pred_1'] = cleaned_results['pred_1'].apply(lambda x: ' '.join(x.split(' ')[:2]))
cleaned_results['pred_2'] = cleaned_results['pred_2'].apply(lambda x: ' '.join(x.split(' ')[:2]))
cleaned_results['pred_3'] = cleaned_results['pred_3'].apply(lambda x: ' '.join(x.split(' ')[:2]))

# Removing \n and periods
cleaned_results['pred_1'] = cleaned_results['pred_1'].str.rstrip('.\n')
cleaned_results['pred_2'] = cleaned_results['pred_2'].str.rstrip('.\n')
cleaned_results['pred_3'] = cleaned_results['pred_3'].str.rstrip('.\n')

cleaned_results[['pred_1', 'pred_2', 'pred_3']].stack().str.lower().value_counts()

,count
confident up,19752
unconfident up,17167
unconfident down,11730
confident down,3047
based on,3


Let's take a look at the three "based on" predictions, and we manually impute it based on the other predictions.

In [ ]:
# "based on" confidence predictions
cleaned_results[(cleaned_results['pred_1'].str.contains('based on')) |
    (cleaned_results['pred_2'].str.contains('based on')) |
    (cleaned_results['pred_3'].str.contains('based on'))]

,Transcript_Num,pred_1,pred_2,pred_3
3254,3254,confident up,based on,based on
10904,10904,based on,confident up,confident up


Let's impute them as "confident up", since that's what the other predictions suggest.

In [ ]:
# Imputing "based on" as "confident up"
cleaned_results = cleaned_results.replace('based on', 'confident up')

cleaned_results[['pred_1', 'pred_2', 'pred_3']].stack().str.lower().value_counts()

,count
confident up,19755
unconfident up,17167
unconfident down,11730
confident down,3047


## EDA on our confidence predictions
We'll first just get a sense of how many predictions of each confidence type there are. Since the order of confidence predictions do not matter, we can transform our confidence labels into features, and then aggregate the number of predictions under each label.

In [ ]:
# Defining the four possible confidence labels
confidence_labels = ['confident up', 'unconfident up', 'unconfident down', 'confident down']

# Creating a function to count occurrences
def count_labels(row):
    counts = {label: 0 for label in confidence_labels}
    for val in [row['pred_1'], row['pred_2'], row['pred_3']]:
        if val in counts:
            counts[val] += 1
    return pd.Series(counts)

# Applying to new df
conf_counts_df = cleaned_results.apply(count_labels, axis=1)

conf_counts_df

,confident up,unconfident up,unconfident down,confident down
0,0,2,1,0
1,2,1,0,0
2,2,0,1,0
3,3,0,0,0
4,2,0,1,0
...,...,...,...,...
17228,2,0,1,0
17229,0,0,3,0
17230,0,1,1,1
17231,1,1,0,1


In [ ]:
# Getting a table of the distribution of confidence labels
conf_counts_df.value_counts()

confident up  unconfident up  unconfident down  confident down
2             1               0                 0                 3024
1             2               0                 0                 2361
3             0               0                 0                 1955
1             1               1                 0                 1882
0             2               1                 0                 1239
2             0               1                 0                 1089
0             1               2                 0                 1037
              3               0                 0                  911
              0               3                 0                  639
1             0               2                 0                  634
0             0               2                 1                  609
              1               1                 1                  491
              0               1                 2                  277
1             0               1                 1                  275
              1               0                 1                  245
0             2               0                 1                  220
              1               0                 2                  115
2             0               0                 1                  105
0             0               0                 3                   68
1             0               0                 2                   57
Name: count, dtype: int64

As expected, most predictions fall under the "up" confidence categories. However, the high number of clusters containing just "up" or "down" labels suggests that the model's outputs are not entirely random and may reflect underlying patterns in the data.

## Creating a simple feedforward model to obtain predictions from confident scores
Now that we've confirmed our EDA, we can create a feedforward model to use the confidence ratings as features (using a subset of the data for training) to predict a direction. Again, the purpose for doing so is because earnings calls are generally positive, and even with the prompt engineering, most of the predictions are going to be upwards.

Therefore, a simple feedforward network should be able to take the three confidence scores and generate a better direction prediction than just a simple average.

*Note: Since we know the transcripts went in order (and we have the transcript number), our target variable (Close_5_dir) will have a 1-to-1 match with each transcript.

In [ ]:
# Using sklearn to split the data
X_train, X_test, y_train, y_test = train_test_split(
    conf_counts_df,
    full_data['Close_5_dir'],
    test_size=0.2,
    random_state=42,
    stratify=full_data['Close_5_dir']
)

In [ ]:
# Creating a simple logistic regression model from sklearn
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced')

# Fitting the data
log_reg.fit(X_train, y_train)

# Predictions
y_pred = log_reg.predict(X_test)

# Getting the classification report
print(classification_report(y_test.values, y_pred))

              precision    recall  f1-score   support

         0.0       0.45      0.46      0.46      1597
         1.0       0.53      0.52      0.53      1850

    accuracy                           0.49      3447
   macro avg       0.49      0.49      0.49      3447
weighted avg       0.49      0.49      0.49      3447



Unfortunately, it appears that the predictions were pretty awful, and the model did not learn well from it. Perhaps due to the number of features, a random forest model might work better.

In [ ]:
# Creating a random forest model from sklearn
rf_model = RandomForestClassifier(class_weight='balanced')

# Fitting the data
rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)

# Getting the classification report
print(classification_report(y_test.values, y_pred_rf))

              precision    recall  f1-score   support

         0.0       0.46      0.29      0.36      1597
         1.0       0.54      0.70      0.61      1850

    accuracy                           0.51      3447
   macro avg       0.50      0.50      0.48      3447
weighted avg       0.50      0.51      0.49      3447



In [ ]:
tree_depth_list = []
for i in range(100):
    tree_depth_list.append(rf_model.estimators_[i].tree_.max_depth)

print("Average tree depth of rf classifier:", sum(tree_depth_list)/len(tree_depth_list))

Average tree depth of rf classifier: 7.6


We'll try grid search to see if we can improve these results.

In [ ]:
# Defining a parameter grid
param_grid = {
    'n_estimators': [100],
    'max_depth': [3, 5, 7],
    'min_samples_split': [5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    rf_model,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Fitting the model
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(class_weight='balanced'),
             n_jobs=-1,
             param_grid={'max_depth': [3, 5, 7], 'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [5, 10, 15, 20],
                         'n_estimators': [100]},
             scoring='accuracy')

In [ ]:
# Predictions
y_pred_gs = grid_search.predict(X_test)

# Getting the classification report
print(classification_report(y_test.values, y_pred_gs))

              precision    recall  f1-score   support

         0.0       0.47      0.42      0.44      1597
         1.0       0.54      0.59      0.57      1850

    accuracy                           0.51      3447
   macro avg       0.51      0.50      0.50      3447
weighted avg       0.51      0.51      0.51      3447



In [ ]:
# Finding the best parameters from grid search
grid_search.best_params_

{'max_depth': 5,
 'min_samples_leaf': 4,
 'min_samples_split': 15,
 'n_estimators': 100}

While random forest did outperform the logistic regression, the grid search barely seemed to do anything, and it appears a random forest model is not good enough to separate these predictions either.

Before we try other models, let's just try making a simple case where we go with baseline majority prediction (up) for the most part, and we only predict negative if all three predictions are confident down (c-down). Essentially, we're assuming the baseline case as up for all predictions, and we just want to see if we can do better than the baseline when there are three c-down predictions. We can also test what happens for the other four labels, sticking with the baseline majority (or minority for the c-up and c-down cases), and only predict the other direction if all three confidence level predictions match.

## Predicting "down" from baseline
As mentioned, we'll have our majority baseline stay as the "up" direction, and if we have a certain threshold of "down" confidence predictions, we'll actually predict down.

### 3 confident downs
To start, we'll see how well our model did for 3 separate "down" predictions.

In [ ]:
# Analyzing accuracy of predictions with 3 separate "confident down" predictions
cdown_3_indices = conf_counts_df[conf_counts_df['confident down'] == 3].index
print('3 separate "confident down" predictions:')
display(model_df.iloc[cdown_3_indices]['Close_5_dir'].value_counts())

3 separate "confident down" predictions:


,count
Close_5_dir,
1,36
0,32


Unfortunately, it appears that we get less than 50% accuracy even when our model generates 3 "confident down" predictions. We can check the other conditions as well, though it's unlikely they will be better as even the most confident down predictions from Llama-3.1 are still incorrect.

### Checking value counts of some other confidence labels

In [ ]:
# Analyzing accuracy of predictions with 3 separate "unconfident down" predictions
udown_3_indices = conf_counts_df[conf_counts_df['unconfident down'] == 3].index
print('3 separate "unconfident down" predictions:')
display(model_df.iloc[udown_3_indices]['Close_5_dir'].value_counts())

# Analyzing accuracy of predictions with 3 separate "unconfident up" predictions
uup_3_indices = conf_counts_df[conf_counts_df['unconfident up'] == 3].index
print('\n3 separate "unconfident up" predictions:')
display(model_df.iloc[uup_3_indices]['Close_5_dir'].value_counts())

# Analyzing accuracy of predictions with 3 separate "confident up" predictions
cup_3_indices = conf_counts_df[conf_counts_df['confident up'] == 3].index
print('\n3 separate "confident up" predictions:')
display(model_df.iloc[cup_3_indices]['Close_5_dir'].value_counts())



3 separate "unconfident down" predictions:


,count
Close_5_dir,
1,350
0,289



3 separate "unconfident up" predictions:


,count
Close_5_dir,
1,469
0,442



3 separate "confident up" predictions:


,count
Close_5_dir,
1,1036
0,919


As expected, it appears most of the predictions hover around 50%, with the "up" predictions staying more prominent.

Therefore, it's clear that no matter what other transformations we perform with the 3 confidence predictions, we'll never perform better than our majority class baseline.

# Comparing saved sentiment results with predictions
Now that we've tested both an encoder and decoder model, we can see if our averaged sentiment actually leads anywhere. Before we do that, however, we saw in our previous chunked FinBERT & Bidirectional LSTM model that getting sentiment and training on it didn't make a difference, and we can confirm that in this next experiment.

Specifically, if we see that our sentiment predictions for the whole transcript do not match with the actual labels, then we can conclude that sentiment models such as FinBERT will not work for this type of prediction.

***Note**: As I mentioned previously, the labels in the following csv are wrong, specifically: 1) 'avg_prob_neg' should be 'avg_prob_pos', 2) 'avg_prob_neu' should be 'avg_prob_neg', and 3) 'avg_prob_pos' should be 'avg_prob_neu'. Basically, everything is shifted over by one spot. Similarly, the class numbers should be: {0: positive, 1: negative, 2: neutral}.

In [ ]:
# Reading in the sentiment results from the previous notebook
finbert_senti_results_path = './finbert_senti_results.csv'
tscrpt_senti_results = pd.read_csv(finbert_senti_results_path, sep='|', index_col='Unnamed: 0')

tscrpt_senti_results

,avg_prob_neg,avg_prob_neu,avg_prob_pos,pred_class,first_chunk_class
0,0.613876,0.060406,0.325719,0,2
1,0.458309,0.138161,0.403530,0,2
2,0.493030,0.087365,0.419605,0,2
3,0.753564,0.027896,0.218540,0,0
4,0.669266,0.057126,0.273608,0,2
...,...,...,...,...,...
3442,0.576985,0.052102,0.370913,0,2
3443,0.393574,0.173594,0.432832,2,2
3444,0.705405,0.074641,0.219954,0,2
3445,0.588332,0.020670,0.390998,0,2


Because we set the same random state and are on the same machine, we know that we'll get the same labels if we run the code from before.

In [ ]:
# Getting matching labels
model_df_20 = model_df.sample(frac=0.20, random_state=42)
model_df_20 = model_df_20.reset_index(drop=True)

Let's keep positive as positive, but let's assume that neutral is more negative (since a non-positive earnings call does likely does not bode as well as a purely positive earnings call).

In [ ]:
# Assume a neutral 2 class means negative (and merging it with the negative 1 class)
tscrpt_senti_results['pred_class'] = np.where(tscrpt_senti_results['pred_class'] == 2, 1,
                                              tscrpt_senti_results['pred_class'])

# Reversing it to match previous labels (1 being up (positive) and 0 being down (negative))
tscrpt_senti_results['pred_class'] = np.abs(1 - tscrpt_senti_results['pred_class'])

# Generating the classification report using only baseline FinBERT sentiment classification
print(classification_report(model_df_20['Close_5_dir'], tscrpt_senti_results['pred_class'], target_names=["Down", "Up"]))

              precision    recall  f1-score   support

        Down       0.50      0.43      0.46      1631
          Up       0.54      0.60      0.57      1816

    accuracy                           0.52      3447
   macro avg       0.52      0.52      0.52      3447
weighted avg       0.52      0.52      0.52      3447



We can also compare our sentiment predictions for the same day results (since the earnings call may have a greater effect on a sooner time period).

In [ ]:
# Testing accuracy when comparing predictions against same day exit at 4:00pm ET Close
model_df_20_same_day_test = model_df_20_same_day.sample(frac=0.20, random_state=42)
model_df_20_same_day_test = model_df_20_same_day_test.reset_index(drop=True)

# Generating the classification report using only baseline FinBERT sentiment classification
print(classification_report(model_df_20_same_day_test['label'], tscrpt_senti_results['pred_class'], target_names=["Down", "Up"]))

              precision    recall  f1-score   support

        Down       0.51      0.42      0.46      1720
          Up       0.51      0.60      0.55      1727

    accuracy                           0.51      3447
   macro avg       0.51      0.51      0.51      3447
weighted avg       0.51      0.51      0.51      3447



But alas, we find that the sentiment does not appear to perform any better, whether the exit occurs days after entry or on the same day. Therefore, it appears that an encoder-only model - at least using sentiment from even domain-specific models lik FinBERT - does not appear to be predictive of stock price drift.

# Combination of Models
Now that I've confirmed my resuls from the FinBERT sentiment classifier - and now that I've also finished testing my Llama model - and I can try a combination (i.e., ensemble) of the two and see what I find.

## Checking ensemble of predictions with drift over time
Lastly, let's check both our encoder (FinBERT) and decoder (Llama-3.1) results and see if it's indicative of up/down drift over time.

Firstly, we'll get our FinBERT results back, and we'll process them the same way as above (just to double-check).

In [6]:
# Reading in the sentiment results from the previous notebook
finbert_senti_results_path = './finbert_senti_results.csv'
tscrpt_senti_results = pd.read_csv(finbert_senti_results_path, sep='|', index_col='Unnamed: 0')

# Assume a neutral 2 class means negative (and merging it with the negative 1 class)
tscrpt_senti_results['pred_class'] = np.where(tscrpt_senti_results['pred_class'] == 2, 1,
                                              tscrpt_senti_results['pred_class'])

# Reversing it to match previous labels (1 being up (positive) and 0 being down (negative))
tscrpt_senti_results['pred_class'] = np.abs(1 - tscrpt_senti_results['pred_class'])

In [12]:
tscrpt_senti_results['pred_class'].value_counts()

,count
pred_class,
1,2021
0,1426


And next, we'll get our llama results, also processing them the same way as we did previously.

In [7]:
# Reading in the llama direction results from the previous notebook
llama_dir_results_path = './llama_direction_results.csv'
llama_dir_results = pd.read_csv(llama_dir_results_path, index_col='Unnamed: 0')

# Getting a cleaned results df and making everything lowercase
cleaned_results = llama_dir_results.copy()
cleaned_results['pred_1'] = cleaned_results['pred_1'].str.lower()
cleaned_results['pred_2'] = cleaned_results['pred_2'].str.lower()
cleaned_results['pred_3'] = cleaned_results['pred_3'].str.lower()

# Truncate to just two words
cleaned_results['pred_1'] = cleaned_results['pred_1'].apply(lambda x: ' '.join(x.split(' ')[:2]))
cleaned_results['pred_2'] = cleaned_results['pred_2'].apply(lambda x: ' '.join(x.split(' ')[:2]))
cleaned_results['pred_3'] = cleaned_results['pred_3'].apply(lambda x: ' '.join(x.split(' ')[:2]))

# Removing \n and periods
cleaned_results['pred_1'] = cleaned_results['pred_1'].str.rstrip('.\n')
cleaned_results['pred_2'] = cleaned_results['pred_2'].str.rstrip('.\n')
cleaned_results['pred_3'] = cleaned_results['pred_3'].str.rstrip('.\n')

# Imputing "based on" as "confident up"
cleaned_results = cleaned_results.replace('based on', 'confident up')

In [8]:
# Defining the four possible confidence labels
confidence_labels = ['confident up', 'unconfident up', 'unconfident down', 'confident down']

# Creating a function to count occurrences
def count_labels(row):
    counts = {label: 0 for label in confidence_labels}
    for val in [row['pred_1'], row['pred_2'], row['pred_3']]:
        if val in counts:
            counts[val] += 1
    return pd.Series(counts)

# Applying to new df
llama_pred_results_full = cleaned_results.apply(count_labels, axis=1)

# Getting a sample using the same method as before (to ensure the same numbers)
llama_pred_results = llama_pred_results_full.sample(frac=0.20, random_state=42)
llama_pred_results = llama_pred_results.reset_index(drop=True)

# Taking a simple majority (up or down)
llama_pred_results['pred_class'] = np.where(llama_pred_results['confident up'] + llama_pred_results['unconfident up'] >= 2, 1, 0)

In [11]:
llama_pred_results['pred_class'].value_counts()

,count
pred_class,
1,2607
0,840


And of course, we'll need to get our labels:

In [9]:
# Getting matching labels for all
model_df_20 = full_data[['Close_0_dir', 'Close_1_dir', 'Close_2_dir', 'Close_5_dir']].copy()
model_df_20 = model_df_20.sample(frac=0.20, random_state=42)
model_df_20 = model_df_20.reset_index(drop=True)

And now we can concatenate all of our results together into one dataframe.

In [10]:
# Concatenating all results into one df
combined_df = pd.concat([tscrpt_senti_results['pred_class'], llama_pred_results['pred_class'], model_df_20], axis=1)
combined_df.columns = ['FinBERT_pred', 'Llama_pred', 'Close_0_dir', 'Close_1_dir', 'Close_2_dir', 'Close_5_dir']

combined_df

,FinBERT_pred,Llama_pred,Close_0_dir,Close_1_dir,Close_2_dir,Close_5_dir
0,1,1,1.0,1.0,1.0,1.0
1,1,1,0.0,1.0,0.0,0.0
2,1,1,1.0,1.0,1.0,1.0
3,1,1,0.0,0.0,0.0,1.0
4,1,1,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...
3442,1,0,0.0,0.0,0.0,0.0
3443,0,1,1.0,1.0,1.0,1.0
3444,1,0,0.0,1.0,1.0,0.0
3445,1,1,0.0,1.0,1.0,1.0


Based on the observations from our previous tests, it definitely seems like both the FinBERT and Llama models predict the positive direction more heavily, which matches what our initial FinBERT (with its chunked sentiment classification) and Llama (with simplified prompting) experiments suggest. Furthermore, this also makes sense with what we know about earnings call transcripts - generally, companies try to make them as positive-sounding as possible to please their investors.

We've already seen how neither model individually does a good job with predicting stock price direction from the transcripts, so let's instead try a combination of the two and see how accurate their predictions are when they agree.

I'll also test their agreement in the positive and negative directions separately, as I want to differentiate how well the model does for each direction.

In [ ]:
# Getting model accuracy only when FinBERT and Llama predictions match the up direction
finbert_eq_llama_cond = (combined_df['FinBERT_pred'] == combined_df['Llama_pred'])
llama_up_cond = (combined_df['Llama_pred'] == 1)
llama_correct_pred_cond = (combined_df['Llama_pred'] == combined_df['Close_5_dir'])

combined_df_up = combined_df[finbert_eq_llama_cond & llama_up_cond]
combined_df_up_correct = combined_df[finbert_eq_llama_cond & llama_up_cond & llama_correct_pred_cond]

# Getting model accuracy only when FinBERT and Llama predictions match the down direction
combined_df_up_down = combined_df[finbert_eq_llama_cond & ~llama_up_cond]
combined_df_up_down_correct = combined_df[finbert_eq_llama_cond & ~llama_up_cond & llama_correct_pred_cond]

print('Accuracy when both FinBERT and Llama predictions are up:', round(len(combined_df_up_correct) / len(combined_df_up), 4))
print('Accuracy when both FinBERT and Llama predictions are down:', round(len(combined_df_up_down_correct) / len(combined_df_up_down), 4))
print('Accuracy of overall majority class baseline:', round(full_data['Close_5_dir'].sum() / len(full_data), 4))
print('Accuracy of test set majority class baseline:', round(combined_df['Close_5_dir'].sum() / len(combined_df), 4))

Accuracy when both FinBERT and Llama predictions are up: 0.546
Accuracy when both FinBERT and Llama predictions are down: 0.5096
Accuracy of overall majority class baseline: 0.5366
Accuracy of test set majority class baseline: 0.5268


And we actually see quite positive results. Specifically, when predicting the Up direction, our combined FinBERT and Llama predictions actually beat both the overall majority class baseline, along with the baseline of just the test set. And while the difference may only be by 1%, a 1% edge in the stock market can be massive. In fact, if we have time, we can test how much money we can make / would have made by comparing what would have happened had we gone in according to our model.

In terms of the Down predictions, the model definitely did not perform as well. However, this is not a surprise as 1) our Down class was smaller in the first place, and 2) that's the difficulty in deciphering earnings calls in the first place.

What this could mean is that the model still struggles to get predict downward trends, likely because it simply is too difficult for a model to pick up on all the underlying nuances in an earnings call. However, it's just as likely that there simply isn't a way to predict stock price movement from earnings calls, simply because the market acts randomly.

In either case, we do at least see that when both the FinBERT and Llama models agree on the Up class, there is indeed a greater likelihood that the model actually produces something positive.

## Checking other time horizon predictions
We've only been focusing on the time horizon of 5 business days (1 week). But let's see how well our predictions do for the other time horizons.

In [ ]:
# Getting the directional comparisons for the other time horizons
combined_df_up_correct_0 = combined_df[finbert_eq_llama_cond & llama_up_cond & (combined_df['Llama_pred'] == combined_df['Close_0_dir'])]
combined_df_up_correct_1 = combined_df[finbert_eq_llama_cond & llama_up_cond & (combined_df['Llama_pred'] == combined_df['Close_1_dir'])]
combined_df_up_correct_2 = combined_df[finbert_eq_llama_cond & llama_up_cond & (combined_df['Llama_pred'] == combined_df['Close_2_dir'])]

combined_df_down_correct_0 = combined_df[finbert_eq_llama_cond & ~llama_up_cond & (combined_df['Llama_pred'] == combined_df['Close_0_dir'])]
combined_df_down_correct_1 = combined_df[finbert_eq_llama_cond & ~llama_up_cond & (combined_df['Llama_pred'] == combined_df['Close_1_dir'])]
combined_df_down_correct_2 = combined_df[finbert_eq_llama_cond & ~llama_up_cond & (combined_df['Llama_pred'] == combined_df['Close_2_dir'])]

print('For time horizon of 2 days:')
print(' - Accuracy when both FinBERT and Llama predictions are up:', round(len(combined_df_up_correct_2) / len(combined_df_up), 4))
print(' - Accuracy when both FinBERT and Llama predictions are down:', round(len(combined_df_down_correct_2) / len(combined_df_up_down), 4))
print(' - Accuracy of overall majority class baseline:', round(full_data['Close_2_dir'].sum() / len(full_data), 4))
print(' - Accuracy of test set majority class baseline:', round(combined_df['Close_2_dir'].sum() / len(combined_df), 4))

print('\nFor time horizon of 1 day:')
print(' - Accuracy when both FinBERT and Llama predictions are up:', round(len(combined_df_up_correct_1) / len(combined_df_up), 4))
print(' - Accuracy when both FinBERT and Llama predictions are down:', round(len(combined_df_down_correct_1) / len(combined_df_up_down), 4))
print(' - Accuracy of overall majority class baseline:', round(full_data['Close_1_dir'].sum() / len(full_data), 4))
print(' - Accuracy of test set majority class baseline:', round(combined_df['Close_1_dir'].sum() / len(combined_df), 4))

print('\nFor time horizon of 0 days (same day exit as entry):')
print(' - Accuracy when both FinBERT and Llama predictions are up:', round(len(combined_df_up_correct_0) / len(combined_df_up), 4))
print(' - Accuracy when both FinBERT and Llama predictions are down:', round(len(combined_df_down_correct_0) / len(combined_df_up_down), 4))
print(' - Accuracy of overall majority class baseline:', 1-round(full_data['Close_0_dir'].sum() / len(full_data), 4))
print(' - Accuracy of test set majority class baseline:', round(combined_df['Close_0_dir'].sum() / len(combined_df), 4))

For time horizon of 2 days:
 - Accuracy when both FinBERT and Llama predictions are up: 0.5303
 - Accuracy when both FinBERT and Llama predictions are down: 0.5
 - Accuracy of overall majority class baseline: 0.5145
 - Accuracy of test set majority class baseline: 0.5138

For time horizon of 1 day:
 - Accuracy when both FinBERT and Llama predictions are up: 0.5184
 - Accuracy when both FinBERT and Llama predictions are down: 0.5
 - Accuracy of overall majority class baseline: 0.5076
 - Accuracy of test set majority class baseline: 0.5138

For time horizon of 0 days (same day exit as entry):
 - Accuracy when both FinBERT and Llama predictions are up: 0.5072
 - Accuracy when both FinBERT and Llama predictions are down: 0.488
 - Accuracy of overall majority class baseline: 0.5065
 - Accuracy of test set majority class baseline: 0.501


And we now obtain our final results, where in all time horizons, the accuracy when the FinBERT and Llama predictions are up is greater than their respective majority class baseline. Furthermore, we also see an interesting trend, where the accuracy seems to increase as the time horizon increases. This could simply indicate that the stocks tend to drift upwards over time (since the majority class also increases), but it could also mean that our model is better at predicting the longer term stock price drift rather than shorter term trends.

This does provide evidence that the way we trained our Llama model is accurate, as we specifically prompted it to predict 1-week results. On the other hand, we used the baseline classifications from FinBERT, with no specific date for the time horizon. Therefore, FinBERT acted like a base from which the Llama predictions could build from, which helped to confirm the predictions from the Llama model (this also likely only worked with positive case because the majority of predictions were positive or neutral).

Either way, we can see from our experiments that predicting the upwards direction performs better than predicting the downwards direction, and if we were to implement our FinBERT-Llama combination model, we should make it a long-only strategy.

# Miscellaneous Analyses
Here are some miscellaneous analyses that couldn't quite fit in seamlessly above.

In [ ]:
# Getting just the FinBERT model's accuracies
finbert_up_cond = (combined_df['FinBERT_pred'] == 1)
finbert_correct_pred_cond = (combined_df['FinBERT_pred'] == combined_df['Close_5_dir'])

# Getting model accuracy only when the FinBERT predictions match the up/down direction
finbert_df_up = combined_df[finbert_up_cond]
finbert_df_up_correct = combined_df[finbert_up_cond & finbert_correct_pred_cond]
finbert_df_down = combined_df[~finbert_up_cond]
finbert_df_down_correct = combined_df[~finbert_up_cond & finbert_correct_pred_cond]

print('Accuracy of FinBERT up predictions:', round(len(finbert_df_up_correct) / len(finbert_df_up), 4))
print('Accuracy of FinBERT down predictions:', round(len(finbert_df_down_correct) / len(finbert_df_down), 4))
print('Accuracy of overall majority class baseline:', round(full_data['Close_5_dir'].sum() / len(full_data), 4))
print('Accuracy of test set majority class baseline:', round(combined_df['Close_5_dir'].sum() / len(combined_df), 4))

Accuracy of FinBERT up predictions: 0.5428
Accuracy of FinBERT down predictions: 0.4958
Accuracy of overall majority class baseline: 0.5366
Accuracy of test set majority class baseline: 0.5268


In [ ]:
# Getting just the Llama model's accuracies
llama_up_cond = (combined_df['Llama_pred'] == 1)
llama_correct_pred_cond = (combined_df['Llama_pred'] == combined_df['Close_5_dir'])

# Getting model accuracy only when the Llama predictions match the up/down direction
llama_df_up = combined_df[llama_up_cond]
llama_df_up_correct = combined_df[llama_up_cond & llama_correct_pred_cond]
llama_df_down = combined_df[~llama_up_cond]
llama_df_down_correct = combined_df[~llama_up_cond & llama_correct_pred_cond]

print('Accuracy of Llama up predictions:', round(len(llama_df_up_correct) / len(llama_df_up), 4))
print('Accuracy of Llama down predictions:', round(len(llama_df_down_correct) / len(llama_df_down), 4))
print('Accuracy of overall majority class baseline:', round(full_data['Close_5_dir'].sum() / len(full_data), 4))
print('Accuracy of test set majority class baseline:', round(combined_df['Close_5_dir'].sum() / len(combined_df), 4))

Accuracy of Llama up predictions: 0.532
Accuracy of Llama down predictions: 0.4893
Accuracy of overall majority class baseline: 0.5366
Accuracy of test set majority class baseline: 0.5268
